# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 220)

print("Libraries loaded.")

Libraries loaded.


In [3]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

PRE_MONTHS = "('2025-07','2025-08','2025-09')"
A_MONTHS = "('2025-10','2025-11','2025-12')"
B_MONTHS = "('2026-01','2026-02','2026-03')"

print("Warehouse connected.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Warehouse connected.


In [4]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "month",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column
    for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Required warehouse columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required warehouse columns are available.


In [5]:
pre = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_pre
    FROM warehouse
    WHERE month IN {PRE_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

feat_a = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS total_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_a,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position,
        ROUND(STDDEV(gsc_avg_position), 2) AS position_volatility,
        ROUND(
            SUM(COALESCE(gsc_clicks, 0)) * 1.0
            / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0),
            4
        ) AS ctr,
        COUNT(
            DISTINCT CASE
                WHEN gsc_data_available IS TRUE
                THEN report_date
            END
        ) AS active_gsc_days
    FROM warehouse
    WHERE month IN {A_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

label_b = con.execute(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_b
    FROM warehouse
    WHERE month IN {B_MONTHS}
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
''').df()

df = feat_a.merge(
    label_b,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

df = df.merge(
    pre,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

df = df[df["total_impressions"] > 0].copy()

df["pct_change"] = (
    (df["clicks_b"] - df["clicks_a"])
    / df["clicks_a"].replace(0, np.nan)
)

df["pct_change_pre_to_a"] = (
    (df["clicks_a"] - df["clicks_pre"])
    / df["clicks_pre"].replace(0, np.nan)
)

def label_row(row):
    if pd.isna(row["pct_change"]):
        return "worth_review"

    was_declining = (
        pd.notna(row["pct_change_pre_to_a"])
        and row["pct_change_pre_to_a"] < -0.10
    )

    if was_declining and row["pct_change"] > 0.05:
        return "recovering"

    if row["pct_change"] > 0.20:
        return "growing"

    if row["pct_change"] < -0.20:
        return "declining"

    return "worth_review"

df["label"] = df.apply(label_row, axis=1)

df["volume_tier"] = pd.qcut(
    df["total_impressions"],
    q=3,
    labels=["low", "med", "high"],
    duplicates="drop",
)

FEATURES = [
    "avg_position",
    "position_volatility",
    "ctr",
    "total_impressions",
    "active_gsc_days",
]

df_model = df.dropna(
    subset=FEATURES + ["label", "volume_tier"]
).copy()

X = pd.get_dummies(
    df_model[FEATURES + ["volume_tier"]],
    columns=["volume_tier"],
    dtype=int,
)

y = df_model["label"].copy()
groups = df_model["client_hash_id"].astype(str)

print("Modeling rows:", len(df_model))
print("Feature matrix:", X.shape)
display(y.value_counts().to_frame("n"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 103933
Feature matrix: (103933, 8)


,n
label,
worth_review,62757
growing,23186
declining,16687
recovering,1303


In [6]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()
y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

train_clients = set(
    df_model.iloc[train_index]["client_hash_id"]
)
test_clients = set(
    df_model.iloc[test_index]["client_hash_id"]
)

assert train_clients.isdisjoint(test_clients)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

print("Validation accuracy:", round(accuracy_score(y_test, predictions), 4))
print(
    "Validation macro F1:",
    round(f1_score(y_test, predictions, average="macro"), 4),
)

display(
    pd.DataFrame(
        classification_report(
            y_test,
            predictions,
            output_dict=True,
            zero_division=0,
        )
    ).T
)

Validation accuracy: 0.8094
Validation macro F1: 0.4418


,precision,recall,f1-score,support
declining,0.592195,0.339106,0.431261,1790.000000
growing,0.256629,0.675810,0.371997,802.000000
recovering,0.000000,0.000000,0.000000,52.000000
worth_review,0.995715,0.933936,0.963837,7962.000000
accuracy,0.809447,0.809447,0.809447,0.809447
macro avg,0.461135,0.487213,0.441774,10606.000000
weighted avg,0.866842,0.809447,0.824474,10606.000000


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks items by:

1. Model confidence
2. Opportunity size, measured by impression percentile
3. Class-specific urgency

Priority order:

- `declining` — highest urgency because measured performance is moving downward
- `recovering` — protect and verify the recovery
- `growing` — protect current momentum
- `worth_review` — inspect manually when confidence or opportunity is meaningful

Each row receives one action label and one main reason code. The reason code explains why the row entered the queue in language a content editor can understand.

In [7]:
scoring_frame = df_model.copy()

full_probabilities = model.predict_proba(X)
full_predictions = model.predict(X)

class_names = list(model.classes_)

probability_columns = [
    f"prob_{class_name}"
    for class_name in class_names
]

probability_frame = pd.DataFrame(
    full_probabilities,
    columns=probability_columns,
    index=scoring_frame.index,
)

scoring_frame = pd.concat(
    [scoring_frame, probability_frame],
    axis=1,
)

scoring_frame["predicted_label"] = full_predictions

scoring_frame["model_confidence"] = (
    scoring_frame[probability_columns].max(axis=1)
)

scoring_frame["impression_percentile"] = (
    scoring_frame["total_impressions"]
    .rank(method="average", pct=True)
)

urgency_weight = {
    "declining": 1.00,
    "recovering": 0.80,
    "growing": 0.65,
    "worth_review": 0.50,
}

scoring_frame["urgency_weight"] = (
    scoring_frame["predicted_label"]
    .map(urgency_weight)
    .astype(float)
)

scoring_frame["action_score"] = (
    100
    * (
        0.55 * scoring_frame["model_confidence"]
        + 0.30 * scoring_frame["impression_percentile"]
        + 0.15 * scoring_frame["urgency_weight"]
    )
).round(2)

print("Scoring frame ready:", len(scoring_frame))

Scoring frame ready: 103933


In [8]:
def assign_action_label(row):
    label = row["predicted_label"]

    if label == "declining":
        return "REVIEW_AND_IMPROVE"

    if label == "recovering":
        return "VERIFY_AND_PROTECT_RECOVERY"

    if label == "growing":
        return "PROTECT_AND_MONITOR"

    return "MANUAL_REVIEW"

def assign_reason_code(row):
    label = row["predicted_label"]

    high_volume = row["impression_percentile"] >= 0.80
    high_confidence = row["model_confidence"] >= 0.70
    weak_ctr = row["ctr"] <= scoring_frame["ctr"].median()
    volatile = (
        row["position_volatility"]
        >= scoring_frame["position_volatility"].median()
    )

    if label == "declining" and high_volume:
        return "HIGH_VOLUME_DECLINE_RISK"

    if label == "declining":
        return "PREDICTED_DECLINE"

    if label == "recovering" and high_confidence:
        return "HIGH_CONFIDENCE_RECOVERY"

    if label == "recovering":
        return "PREDICTED_RECOVERY"

    if label == "growing" and high_volume:
        return "HIGH_VOLUME_GROWTH"

    if label == "growing":
        return "PREDICTED_GROWTH"

    if weak_ctr and volatile:
        return "WEAK_CTR_POSITION_VOLATILITY"

    if weak_ctr:
        return "WEAK_CTR_REVIEW"

    return "GENERAL_MANUAL_REVIEW"

scoring_frame["action_label"] = scoring_frame.apply(
    assign_action_label,
    axis=1,
)

scoring_frame["reason_code"] = scoring_frame.apply(
    assign_reason_code,
    axis=1,
)

queue = scoring_frame.sort_values(
    by=[
        "action_score",
        "total_impressions",
    ],
    ascending=[
        False,
        False,
    ],
).reset_index(drop=True)

queue.insert(
    0,
    "rank",
    range(1, len(queue) + 1),
)

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "predicted_label",
    "model_confidence",
    "reason_code",
    "action_label",
    "total_impressions",
    "clicks_a",
    "ctr",
    "avg_position",
    "position_volatility",
    "active_gsc_days",
    "volume_tier",
]

queue = queue[queue_columns]

display(queue.head(25))

KeyboardInterrupt: 

In [ ]:
action_summary = (
    queue
    .groupby(
        ["predicted_label", "action_label", "reason_code"],
        observed=True,
    )
    .agg(
        n=("content_hash_id", "size"),
        median_action_score=("action_score", "median"),
        median_confidence=("model_confidence", "median"),
        median_impressions=("total_impressions", "median"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)

display(action_summary)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended users

- Content editors
- SEO analysts
- Content operations leads
- Analysts reviewing portfolio-level opportunities

### Intended use

The queue helps teams decide which anonymized content items deserve human attention first. It supports prioritization, triage, and review planning.

### Valid use

- Rank limited editorial capacity
- Identify high-confidence decline risks
- Protect high-value growing or recovering pages
- Select pages for manual investigation
- Compare queue composition over time

### Limits

- The model predicts trajectory classes, not causal impact.
- A `declining` prediction does not prove that content quality is the cause.
- A `growing` prediction does not guarantee continued growth.
- A `recovering` prediction is lower confidence when the class is rare.
- The model does not observe query intent, SERP features, content text, seasonality, or business value.
- Predictions should not be used outside a similar GSC data context without validation.

In [ ]:
intended_use = pd.DataFrame([
    {
        "item": "Primary user",
        "value": "Content editor or SEO analyst",
    },
    {
        "item": "Primary decision",
        "value": "Which content items should be reviewed first?",
    },
    {
        "item": "Output type",
        "value": "Ranked decision-support queue",
    },
    {
        "item": "Not supported",
        "value": "Automatic editing, deletion, or causal traffic claims",
    },
    {
        "item": "Required safeguard",
        "value": "Human review before action",
    },
])

display(intended_use)

limits = pd.DataFrame([
    {
        "limit": "Observational labels",
        "why_it_matters": (
            "The model learns later click trajectories, not the effect "
            "of an editorial intervention."
        ),
    },
    {
        "limit": "Missing intent context",
        "why_it_matters": (
            "CTR and ranking behavior may be explained by query intent "
            "or SERP layout."
        ),
    },
    {
        "limit": "Class imbalance",
        "why_it_matters": (
            "Rare classes, especially recovering, may have weaker recall."
        ),
    },
    {
        "limit": "Time dependence",
        "why_it_matters": (
            "Seasonality and search demand can change after the training window."
        ),
    },
    {
        "limit": "No business-value feature",
        "why_it_matters": (
            "Traffic opportunity is not the same as commercial importance."
        ),
    },
])

display(limits)

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A reviewer must check context before acting.

### Required human review

- Confirm that the page still exists and remains relevant.
- Review the current title, description, and main content.
- Check whether search intent has changed.
- Check seasonality and recent demand changes.
- Inspect SERP features and strong competitors.
- Confirm that tracking coverage is adequate.
- Compare the predicted class with recent raw performance.
- Record the final decision and rationale.

### No-go list

The following actions must never be automated from this queue alone:

- Delete or unpublish content
- Rewrite titles or body content automatically
- Change canonical tags, redirects, or indexing settings
- Promise traffic growth
- Contact a client
- Allocate budget
- Treat hashed identifiers as personal identities
- Publish client-level findings

In [ ]:
review_checklist = pd.DataFrame([
    {
        "review_step": 1,
        "question": "Is the page still active and strategically relevant?",
        "required": True,
    },
    {
        "review_step": 2,
        "question": "Is the model confidence high enough to prioritize review?",
        "required": True,
    },
    {
        "review_step": 3,
        "question": "Could seasonality or demand explain the trajectory?",
        "required": True,
    },
    {
        "review_step": 4,
        "question": "Could search intent or SERP features explain CTR?",
        "required": True,
    },
    {
        "review_step": 5,
        "question": "Is GSC coverage sufficient and stable?",
        "required": True,
    },
    {
        "review_step": 6,
        "question": "Does the current content actually need a change?",
        "required": True,
    },
    {
        "review_step": 7,
        "question": "Has the reviewer documented the final decision?",
        "required": True,
    },
])

no_go_list = pd.DataFrame([
    {"prohibited_action": "Delete or unpublish automatically"},
    {"prohibited_action": "Rewrite content automatically"},
    {"prohibited_action": "Change redirects or canonical tags automatically"},
    {"prohibited_action": "Promise traffic growth"},
    {"prohibited_action": "Contact clients from model output"},
    {"prohibited_action": "Expose names, URLs, domains, or private queries"},
])

display(review_checklist)
display(no_go_list)

In [ ]:
top20_review = queue.head(20).copy()

top20_review["human_review_note"] = top20_review.apply(
    lambda row: (
        f"Review {row['predicted_label']} recommendation with "
        f"{row['model_confidence']:.1%} confidence and "
        f"{int(row['total_impressions']):,} observed impressions. "
        f"Reason: {row['reason_code']}. Check intent, seasonality, "
        f"SERP context, tracking coverage, and current content before acting."
    ),
    axis=1,
)

top20_review["what_would_make_it_wrong"] = (
    "The recommendation may be wrong if demand changed, tracking is incomplete, "
    "the page serves a different intent, SERP features changed, or the content "
    "has already been updated."
)

display(
    top20_review[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action_label",
            "reason_code",
            "human_review_note",
            "what_would_make_it_wrong",
        ]
    ]
)

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The action playbook can become stale when the data distribution, label distribution, model performance, or business process changes.

### Monitor each scoring cycle

- Number of eligible rows
- Class distribution
- Confidence distribution
- Feature medians
- Missing-data rates
- Queue size by action
- Human acceptance rate
- Precision of reviewed high-priority items
- Per-class recall when later labels become available

### Retrain triggers

Retrain or revalidate when any of the following occurs:

- More than 20% relative change in a major feature median
- More than 10 percentage-point change in predicted class share
- Macro F1 decreases by more than 0.05
- `declining` recall decreases by more than 0.10
- GSC coverage drops materially
- The feature or label definition changes
- The model is older than one quarter
- New clients or content types differ materially from training data

In [ ]:
training_reference = {
    "row_count": int(len(df_model)),
    "class_distribution": (
        y.value_counts(normalize=True)
        .sort_index()
        .to_dict()
    ),
    "feature_medians": {
        feature: float(X[feature].median())
        for feature in X.columns
    },
    "confidence_median": float(
        scoring_frame["model_confidence"].median()
    ),
}

monitoring_triggers = pd.DataFrame([
    {
        "metric": "Major feature median",
        "trigger": ">20% relative shift",
        "response": "Investigate drift and revalidate",
    },
    {
        "metric": "Predicted class share",
        "trigger": ">10 percentage-point shift",
        "response": "Check data and demand changes",
    },
    {
        "metric": "Macro F1",
        "trigger": "Drop >0.05",
        "response": "Retrain or revise features",
    },
    {
        "metric": "Declining recall",
        "trigger": "Drop >0.10",
        "response": "Pause high-risk automation and retrain",
    },
    {
        "metric": "GSC availability",
        "trigger": "Material coverage decline",
        "response": "Pause scoring for affected rows",
    },
    {
        "metric": "Model age",
        "trigger": ">1 quarter",
        "response": "Run scheduled validation",
    },
])

display(monitoring_triggers)

In [ ]:
current_monitoring_snapshot = {
    "row_count": int(len(queue)),
    "predicted_class_distribution": (
        queue["predicted_label"]
        .value_counts(normalize=True)
        .sort_index()
        .to_dict()
    ),
    "median_action_score": float(
        queue["action_score"].median()
    ),
    "median_model_confidence": float(
        queue["model_confidence"].median()
    ),
    "high_priority_rows": int(
        (queue["action_score"] >= 80).sum()
    ),
}

display(
    pd.Series(
        current_monitoring_snapshot,
        name="value",
    ).to_frame()
)

In [ ]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queue_path = OUTPUT_DIR / "action_playbook_queue.csv"
top20_path = OUTPUT_DIR / "action_playbook_top20.csv"
counts_path = OUTPUT_DIR / "action_playbook_action_counts.csv"
summary_path = OUTPUT_DIR / "action_playbook_summary.json"

queue.to_csv(
    queue_path,
    index=False,
)

top20_review.to_csv(
    top20_path,
    index=False,
)

action_counts = (
    queue
    .groupby(
        ["predicted_label", "action_label", "reason_code"],
        observed=True,
    )
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

action_counts.to_csv(
    counts_path,
    index=False,
)

summary = {
    "assignment": "ML-10 content action playbook",
    "lane": "Refresh / Content Opportunity Scoring",
    "model": "RandomForestClassifier",
    "feature_window": "2025-10 to 2025-12",
    "label_window": "2026-01 to 2026-03",
    "queue_rows": int(len(queue)),
    "class_counts": (
        queue["predicted_label"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "action_counts": (
        queue["action_label"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "reason_code_counts": (
        queue["reason_code"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "median_action_score": float(
        queue["action_score"].median()
    ),
    "median_confidence": float(
        queue["model_confidence"].median()
    ),
    "human_review_required": True,
    "automatic_content_changes_allowed": False,
    "private_fields_used": False,
    "future_inputs_used_as_scoring_features": False,
    "training_reference": training_reference,
    "monitoring_snapshot": current_monitoring_snapshot,
}

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(
        summary,
        file,
        indent=2,
        default=str,
    )

print("Queue:", queue_path)
print("Top 20 review:", top20_path)
print("Action counts:", counts_path)
print("Summary:", summary_path)

In [ ]:
forbidden_output_terms = [
    "client_name",
    "company_name",
    "domain",
    "url",
    "query",
    "keyword",
    "page_title",
    "content_text",
]

private_output_hits = [
    column
    for column in queue.columns
    if any(
        term in column.lower()
        for term in forbidden_output_terms
    )
]

required_queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "predicted_label",
    "model_confidence",
    "reason_code",
    "action_label",
]

missing_queue_columns = [
    column
    for column in required_queue_columns
    if column not in queue.columns
]

print("Private output hits:", private_output_hits)
print("Missing queue columns:", missing_queue_columns)

assert private_output_hits == []
assert missing_queue_columns == []
assert queue["action_score"].is_monotonic_decreasing
assert queue_path.exists()
assert top20_path.exists()
assert counts_path.exists()
assert summary_path.exists()

print("PASS: Export and privacy checks completed.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
assert train_clients.isdisjoint(test_clients)
assert private_output_hits == []
assert missing_queue_columns == []
assert queue["action_score"].is_monotonic_decreasing
assert summary_path.exists()

print("ML-10 COMPLETE")
print("Notebook: work/notebooks/w07_action_playbook.ipynb")
print("Queue rows:", len(queue))
print("Summary receipt:", summary_path)